In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [9]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, row_number
from pyspark.sql.window import Window
import pandas as pd

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5-Mandiri") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Bikin df_target
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# 3. Baca dari file CSV lokal & tambah kolom pendapatan
df_transaksi = spark.read.csv("transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

In [18]:
#A

ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner")
hasil_a = hasil_a.withColumn(
    "pencapaian_persen", 
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil_a.orderBy(col("pencapaian_persen").desc()).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [20]:
#B

pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("pendapatan_kategori")
)

window_b = Window.partitionBy("kota").orderBy(col("pendapatan_kategori").desc())

hasil_b = pendapatan_kategori.withColumn("peringkat", row_number().over(window_b))
hasil_b.filter(col("peringkat") == 1).drop("peringkat").show()

+----------+--------------------+-------------------+
|      kota|            kategori|pendapatan_kategori|
+----------+--------------------+-------------------+
|  Magelang|Kesehatan & Kecan...|            7275000|
| Purworejo|Kesehatan & Kecan...|           10075000|
|  Semarang|        Rumah Tangga|           11125000|
|      Solo|Kesehatan & Kecan...|            8425000|
|Yogyakarta|             Fashion|           13325000|
+----------+--------------------+-------------------+



In [25]:
#C

df_transaksi.createOrReplaceTempView("transaksi_tugas")
df_target.createOrReplaceTempView("target_tugas")

hasil_c = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi_tugas t
    JOIN target_tugas tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



Berdasarkan hasil analisis data transaksi dan target bulanan cabang, cabang dengan kinerja paling baik adalah Purworejo di bawah pimpinan PIC Fitri. Cabang ini berhasil melampaui target bulanan secara signifikan dengan tingkat pencapaian sebesar 152,17% (total pendapatan Rp45.650.000 dari target Rp30.000.000). Performa tinggi ini didorong oleh kontribusi utama dari kategori produk unggulan mereka.

Sebaliknya, cabang yang paling membutuhkan perhatian khusus dari manajemen adalah Semarang dengan PIC Sari. Cabang Semarang mencatat persentase pencapaian terendah di antara seluruh cabang, yaitu hanya sebesar 69,41% (total pendapatan Rp38.175.000 dari target Rp55.000.000). Manajemen disarankan melakukan evaluasi strategi pemasaran dan distribusi stok di cabang Semarang, serta mengadopsi taktik promosi dari cabang Purworejo untuk mendongkrak penjualan di periode mendatang.
